In [1]:
from datetime import datetime, date
import asyncio
from ib_insync import IB, Stock, util
import xlwings as xw
import os

from Class_xlWings import *
xlw  = xlWings()

In [2]:
def file_maintenance(xlApp, save_date):
    core_path = r"C:\Users\micha\Market_Data_Pulls"
    core_filename = "ETF_"
    filename_ext = ".xlsx"

    open_filename = core_filename + "Template" + filename_ext
    open_address = os.path.join(core_path, open_filename)

    wb = xlApp.books.open(open_address)

    save_filename = core_filename + save_date.strftime('%Y%m%d') + filename_ext
    save_address = os.path.join(core_path, save_filename)

    wb.save(save_address)

    rtn_filename = os.path.basename(save_address)

    return wb, rtn_filename

In [3]:
async def get_etf_snapshot(ib, symbol="IBIT"):
    stock = Stock(symbol, "SMART", "USD")
    await ib.qualifyContractsAsync(stock)

    ticker = ib.reqMktData(stock, "", snapshot=True)

    # Important: use ib.sleep not asyncio.sleep
    await asyncio.sleep(1)

    df = pd.DataFrame([{
        "symbol": symbol,
        "mkt_bid_price": ticker.bid,
        "mkt_bid_size": ticker.bidSize,
        "mkt_ask_price": ticker.ask,
        "mkt_ask_size": ticker.askSize,
    }])

    return df     # <-- MUST return

In [4]:
current_date_nyc = date.today()
current_time_nyc = str(datetime.now().time())

channel = int(current_time_nyc[0:2] + current_time_nyc[3:5] + current_time_nyc[6:8]) 

# ---- Main async flow ----
async def main():
    ib = IB()
    xlApp = None
    wb = None

    try:
        await ib.connectAsync("127.0.0.1", 7496, clientId=channel)

        xlApp = xw.App(visible=False)
        wb, save_filename = file_maintenance(xlApp, current_date_nyc)

        etf_list = [
            'ARKB', 'BITB', 'BRRR', 'BTC', 'BTCW',
            'EZBC', 'FBTC', 'GBTC', 'IBIT'
        ]

        # Fetch all snapshots concurrently
        rows = await asyncio.gather(
            *(get_etf_snapshot(ib, etf) for etf in etf_list)
        )
        
        df = pd.concat(rows, ignore_index=True)
        df.insert(0, 'ETF', etf_list)

        xlw.printDFToXL(save_filename, 'MKT_DATA', 'A1', df)

        wb.save()

    finally:
        # ---- Guaranteed cleanup ----
        if wb is not None:
            wb.close()

        if xlApp is not None:
            xlApp.quit()

        if ib.isConnected():
            ib.disconnect()


# ---- Run ----
await main()